# Multi-buffer cat: maximal Lyapunov exponent from the origin (12/10/2026)

Lean rerun of the Lyapunov cells of `test_multi_buffer.ipynb` (same EOMs, copied below) with the article conventions:
$g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (was $J_1$) and $\kappa_b = \pi/10.4$ (was $1/10.4$), Julia damping convention.
Each of the $N$ ATS carries $E_J/N$, so $g_2^\mathrm{eff} = \sum_n g_{n} = g_2$. Each buffer is driven at $\varepsilon_d/N$, with
$\varepsilon_d = 10.12\,g_2$: the RWA fixed points $a^2 = -(\varepsilon_d/N)/(2g_2/N)$ then give the same cat size $|\alpha|^2 = \varepsilon_d/2g_2$ for every $N$
(with $\varepsilon_d$ on every buffer, the cat would be $N$ times larger).

Initial state: all modes at the origin, with a state vector of the right size for each model (4, 6, 8); the former notebook
passed the 8-dimensional tri-buffer vector to all three models.

Outputs: `data/variants/lyapunov_multi_buffer_12_10_2026.json` and `figures/variants/lyapunov_multi_buffer_12_10_2026.pdf`.

In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
using ChaosTools, JSON

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b=π/10.4
κ_a = 0

ϵ_p_of(g2_MHz) = asin(g2_MHz*2π*1e-3/(E_J*φ_a^2*φ_b))   # inverse of g2 = E_J sin(ϵ_p) φ_a² φ_b (static pump, ω_p = 0)

In [ ]:
#ATS + Mean_Field
function equation_1ODE_bibuffer!(du,u,p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    ϕ_a = u[1]
    ϕ_b_1 = u[2]
    ϕ_b_2 = u[3]
    ϕ_p_a = u[4]
    ϕ_p_b_1 = u[5]
    ϕ_p_b_2 = u[6]

    #coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_2 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)/2
        
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b_1
    du[3] = ϕ_p_b_2
    du[4] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*(coupling+coupling_2)) - 2 * p.environment_prepared.κ_a * du[1]
    du[5] = -ω_b*(ω_b*ϕ_b_1 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    du[6] = -ω_b*(ω_b*ϕ_b_2 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_2) - 2 * p.environment_prepared.κ_b * du[3]
    return nothing
end

#ATS + Mean_Field
function equation_1ODE_tribuffer!(du,u,p::SimulationParams{SystemATS_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    ϕ_a = u[1]
    ϕ_b_1 = u[2]
    ϕ_b_2 = u[3]
    ϕ_b_3 = u[4]
    ϕ_p_a = u[5]
    ϕ_p_b_1 = u[6]
    ϕ_p_b_2 = u[7]
    ϕ_p_b_3 = u[8]

    #coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + (ϕ_b_1 + ϕ_b_2 + ϕ_b_3) * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_1 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_2 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_2 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    coupling_3 = (cos(ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b_3 * sqrt(Z_b)/ϕ_quantum) - 1) / ϕ_quantum
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)/3
        
    du[1] = ϕ_p_a
    du[2] = ϕ_p_b_1
    du[3] = ϕ_p_b_2
    du[4] = ϕ_p_b_3
    du[5] = -ω_a*(ω_a*ϕ_a - sqrt(Z_a) * sin_pump*(coupling+coupling_2+coupling_3)) - 2 * p.environment_prepared.κ_a * du[1]
    du[6] = -ω_b*(ω_b*ϕ_b_1 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling) - 2 * p.environment_prepared.κ_b * du[2]
    du[7] = -ω_b*(ω_b*ϕ_b_2 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_2) - 2 * p.environment_prepared.κ_b * du[3]
    du[8] = -ω_b*(ω_b*ϕ_b_3 + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase)*sqrt(2) - sqrt(Z_b) * sin_pump*coupling_3) - 2 * p.environment_prepared.κ_b * du[4]
    return nothing
end

In [ ]:
t_max = 30/κ_b
t_w = 15/κ_b
system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, 0.0, 0.0, 0.0, 0.0)
environment = MeanField(0.0, κ_b)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment)
dt = get_dt(system, environment)

ϵ_p_list = vcat(0.2:.01:1.5)
ϵ_d_g2_ratio = 10.12
u_0_mono, u_0_bi, u_0_tri = zeros(4), zeros(6), zeros(8)   # (ϕ, ϕ̇) of the memory and of each buffer, all at the origin

In [ ]:
Lyap_mono = zeros(length(ϵ_p_list))
Lyap_bi = zeros(length(ϵ_p_list))
Lyap_tri = zeros(length(ϵ_p_list))
Threads.@threads for i in eachindex(ϵ_p_list)
    ϵ_p_i = ϵ_p_list[i]
    g2_i = sin(ϵ_p_i)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
    # N buffers: each one driven at ϵ_d/N, so that the cat size ϵ_d/2g2 does not depend on N
    p_N(N) = SimulationParams(system_prepared, prepare(DriveTwoModesBasics(ϵ_d_g2_ratio*g2_i/N, ω_d, buffer_phase, ϵ_p_i, ω_p, pump_phase)), environment_prepared)
    Lyap_mono[i] = lyapunov(CoupledODEs(equation_1ODE!, u_0_mono, p_N(1)), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    Lyap_bi[i] = lyapunov(CoupledODEs(equation_1ODE_bibuffer!, u_0_bi, p_N(2)), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
    Lyap_tri[i] = lyapunov(CoupledODEs(equation_1ODE_tribuffer!, u_0_tri, p_N(3)), t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
end

In [ ]:
g2_list = E_J.*sin.(ϵ_p_list).*φ_a^2 .*φ_b/2π*1e3
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/variants/lyapunov_multi_buffer_12_10_2026.json", "w") do io
    JSON.print(io, Dict("eps_p" => ϵ_p_list, "g2_eff_MHz" => g2_list, "eps_d_over_g2" => ϵ_d_g2_ratio, "drive_per_buffer" => "eps_d/N", "kappa_b" => κ_b,
                        "lyap_mono" => Lyap_mono, "lyap_bi" => Lyap_bi, "lyap_tri" => Lyap_tri))
end
# first g2 with a positive exponent, for each model
for (name, λ) in (("mono", Lyap_mono), ("bi", Lyap_bi), ("tri", Lyap_tri))
    i = findfirst(>(0), λ)
    println(name, "-buffer: first positive exponent at g2_eff/2π = ", isnothing(i) ? "none" : round(g2_list[i], digits=1), " MHz")
end

In [ ]:
plot(g2_list, Lyap_mono, label = "Mono-buffer", color = RGB(0, 0, 0))
plot!(g2_list, Lyap_bi, label = "Bi-buffer", color = RGB(134/255, 187/255, 216/255))
plot!(g2_list, Lyap_tri, label = "Tri-buffer", color = RGB(160/255, 40/255, 41/255))
hline!([0], color = :gray, ls = :dash, label = "")
plot!(xlabel = L"g_2^\mathrm{eff}/2\pi"*" (MHz)", ylabel = L"\lambda"*" (ns"*L"^{-1}"*")", legend = :topleft)
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/variants/lyapunov_multi_buffer_12_10_2026.pdf")